<a href="https://colab.research.google.com/github/shimataiyaki/digital-ema-app/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

digital-ema Analysis by Python

Step 1: データ読み込み（CSVアップロード）

Step 2: ランダム方式シミュレーション（100回 + 95%信頼区間）

Step 3: キュー方式シミュレーション（1回）

Step 4: 来場者体験分析（滞在時間別表示確率）＋ 文字サマリー

Step 5: CSV一括出力（2ファイル）

In [ ]:
import pandas as pd
from google.colab import files
import io

print("CSVファイルをアップロードしてください")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
timestamp_col = df.columns[0]
df['datetime'] = pd.to_datetime(df[timestamp_col])
df = df.sort_values('datetime').reset_index(drop=True)

N = len(df)
print(f"データ件数: {N}件")
print(f"期間: {df['datetime'].min()} ～ {df['datetime'].max()}")

CSVファイルをアップロードしてください


Saving kounai.csv to kounai (1).csv
データ件数: 200件
期間: 2026-06-05 13:17:32 ～ 2026-06-05 16:12:56


In [ ]:
import numpy as np
import random
from datetime import timedelta

NUM_SIMULATIONS = 100
DISPLAY_COUNT = 15
INTERVAL_SEC = 20
MAX_WAIT_MINUTES = 10

start_time = df['datetime'].min()
end_time = df['datetime'].max() + timedelta(minutes=10)
time_range = pd.date_range(start=start_time, end=end_time, freq='1s')

cumulative_rates_all = np.zeros((NUM_SIMULATIONS, MAX_WAIT_MINUTES + 1))
all_wait_times_random = []

for sim in range(NUM_SIMULATIONS):
    if sim % 10 == 0:
        print(f"  Random: {sim}/{NUM_SIMULATIONS}")

    displayed_set = set()
    first_display_time = {}
    arrived_indices = set()       # 到着済み投稿の管理（修正箇所1）

    for current_time in time_range:
        # ① 到着済み投稿を更新（修正箇所2：完全一致ではなく「時刻を過ぎたか」で判定）
        new_arrivals = df[df['datetime'] <= current_time].index.tolist()
        arrived_indices.update(new_arrivals)

        # ② 20秒ごとに表示を更新
        if current_time.second % INTERVAL_SEC == 0 and current_time != start_time:
            available = list(arrived_indices)    # 表示候補は到着済み投稿のみ

            if len(available) >= DISPLAY_COUNT:
                selected = random.sample(available, DISPLAY_COUNT)
            elif len(available) > 0:
                selected = available[:]
            else:
                selected = []

            for idx in selected:
                if idx not in displayed_set:
                    displayed_set.add(idx)
                    arrival = df.loc[idx, 'datetime']
                    wait_sec = (current_time - arrival).total_seconds()
                    # バリデーション（修正箇所3）
                    if wait_sec < 0:
                        wait_sec = 0
                    first_display_time[idx] = wait_sec

    wait_values = list(first_display_time.values())
    all_wait_times_random.extend(wait_values)

    for minute in range(MAX_WAIT_MINUTES + 1):
        threshold = minute * 60
        count = sum(1 for w in wait_values if w <= threshold)
        cumulative_rates_all[sim, minute] = count / N * 100

mean_rates_random = np.mean(cumulative_rates_all, axis=0)
lower_ci_random = np.percentile(cumulative_rates_all, 2.5, axis=0)
upper_ci_random = np.percentile(cumulative_rates_all, 97.5, axis=0)

print("ランダム方式シミュレーション完了")
print(f"平均待ち時間: {np.mean(all_wait_times_random):.1f}秒")
print(f"最大待ち時間: {np.max(all_wait_times_random):.1f}秒")

  Random: 0/100
  Random: 10/100
  Random: 20/100
  Random: 30/100
  Random: 40/100
  Random: 50/100
  Random: 60/100
  Random: 70/100
  Random: 80/100
  Random: 90/100
ランダム方式シミュレーション完了
平均待ち時間: 125.4秒
最大待ち時間: 1824.0秒


In [ ]:
from collections import deque

queue = deque()
displayed_set_queue = set()
first_display_time_queue = {}
arrived_indices_queue = set()

for current_time in time_range:
    # 到着済み投稿を更新
    new_arrivals = df[df['datetime'] <= current_time].index.tolist()
    newly_arrived = [idx for idx in new_arrivals if idx not in arrived_indices_queue]
    for idx in newly_arrived:
        queue.append(idx)
        arrived_indices_queue.add(idx)

    # 20秒ごとに表示を更新
    if current_time.second % INTERVAL_SEC == 0 and current_time != start_time:
        queue_display = []
        while len(queue_display) < DISPLAY_COUNT and len(queue) > 0:
            idx = queue.popleft()
            queue_display.append(idx)

        remaining = DISPLAY_COUNT - len(queue_display)
        if remaining > 0 and N > len(queue_display):
            already_shown = set(queue_display) | displayed_set_queue
            candidates = [i for i in range(N) if i not in already_shown and i in arrived_indices_queue]
            if len(candidates) > 0:
                random_fill = random.sample(candidates, min(remaining, len(candidates)))
                queue_display.extend(random_fill)

        for idx in queue_display:
            if idx not in displayed_set_queue:
                displayed_set_queue.add(idx)
                arrival = df.loc[idx, 'datetime']
                wait_sec = (current_time - arrival).total_seconds()
                first_display_time_queue[idx] = max(0, wait_sec)

wait_times_queue = list(first_display_time_queue.values())
print("ハイブリッド方式シミュレーション完了")
print(f"平均待ち時間: {np.mean(wait_times_queue):.1f}秒")
print(f"最大待ち時間: {np.max(wait_times_queue):.1f}秒")

ハイブリッド方式シミュレーション完了
平均待ち時間: 10.1秒
最大待ち時間: 19.0秒


In [ ]:
def calc_find_probability(wait_list, stay_minutes=[1, 2, 3, 5, 10]):
    results = []
    for stay_min in stay_minutes:
        threshold = stay_min * 60
        count = sum(1 for w in wait_list if w <= threshold)
        prob = count / len(wait_list) * 100
        results.append({
            '滞在時間(分)': stay_min,
            '見つけられる確率(%)': round(prob, 1),
            '見つけられる人数': count,
            '総人数': len(wait_list)
        })
    return results

random_find = calc_find_probability(all_wait_times_random)
hybrid_find = calc_find_probability(wait_times_queue)

print("■ 来場者体験分析")
print("ランダム方式:")
for r in random_find:
    print(f"  {r['滞在時間(分)']}分待てば {r['見つけられる確率(%)']}% の人が見つけられる")
print("ハイブリッド方式:")
for r in hybrid_find:
    print(f"  {r['滞在時間(分)']}分待てば {r['見つけられる確率(%)']}% の人が見つけられる")

■ 来場者体験分析
ランダム方式:
  1分待てば 48.6% の人が見つけられる
  2分待てば 66.3% の人が見つけられる
  3分待てば 76.8% の人が見つけられる
  5分待てば 88.2% の人が見つけられる
  10分待てば 97.9% の人が見つけられる
ハイブリッド方式:
  1分待てば 100.0% の人が見つけられる
  2分待てば 100.0% の人が見つけられる
  3分待てば 100.0% の人が見つけられる
  5分待てば 100.0% の人が見つけられる
  10分待てば 100.0% の人が見つけられる


In [ ]:
import csv

# 5-1. 累積表示率（信頼区間付き）
with open('cumulative_rates_with_ci.csv', 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['待ち時間(分)', 'ランダム再現値_平均(%)', 'ランダム_下限95%CI', 'ランダム_上限95%CI', 'ハイブリッド再現値(%)'])
    for minute in range(MAX_WAIT_MINUTES + 1):
        hybrid_rate = sum(1 for w in wait_times_queue if w <= minute * 60) / N * 100
        writer.writerow([
            minute,
            round(mean_rates_random[minute], 1),
            round(lower_ci_random[minute], 1),
            round(upper_ci_random[minute], 1),
            round(hybrid_rate, 1)
        ])

# 5-2. 来場者体験分析
with open('find_probability_by_stay_time.csv', 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['方式', '滞在時間(分)', '見つけられる確率(%)', '見つけられる人数', '総人数'])
    for r in random_find:
        writer.writerow(['ランダム方式', r['滞在時間(分)'], r['見つけられる確率(%)'], r['見つけられる人数'], r['総人数']])
    for r in hybrid_find:
        writer.writerow(['ハイブリッド方式', r['滞在時間(分)'], r['見つけられる確率(%)'], r['見つけられる人数'], r['総人数']])

files.download('cumulative_rates_with_ci.csv')
files.download('find_probability_by_stay_time.csv')
print("CSVファイルをダウンロードしました")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CSVファイルをダウンロードしました
